# Task 1

## Set up Ollama

In [7]:
!ollama pull llama3.2

]11;?\[GIN] 2026/02/28 - 18:01:52 | 200 |     128.917µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ [GIN] 2026/02/28 - 18:01:52 | 200 |  566.412834ms |       127.0.0.1 | POST     "/api/pull"
pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
from openai import OpenAI


# Create client pointing to Ollama

client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'  # Required by SDK, but not used by Ollama

)


response = client.chat.completions.create(
    model='llama3.2',
    messages=[{'role': 'system', 'content': 'You are a helpful assistant.'},
              {'role': 'user', 'content': 'What is 2+2?'}]
              )


print(response.choices[0].message.content)

ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 10.738 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   Apple M3 Pro
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal3  (5001)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSetSize  = 12884.92 MB
llama_model_load_from_file_impl: using device Metal (Apple M3 Pro) 

[GIN] 2026/02/28 - 18:03:21 | 200 | 26.447568375s |       127.0.0.1 | POST     "/v1/chat/completions"
2 + 2 = 4


## Running the programs

In [ ]:
# To calculate the time, I ran the programs through terminal 
# and used the "Duration" as calculated in the program which calculates the difference between the start and end time for running the model for evaluation (see main()).

############ Observations
# Running tasks one at a time (Sequential): 
# Executing the programs one after the other is faster without Ollama. Native execution took only 1.3 minutes, compared to 1.7 minutes with Ollama.

# Running tasks at the same time (Parallel): 
# When executing both programs simultaneously, Ollama handles the workload much better. 
# Ollama finished in 1.5 minutes, whereas native execution slowed down significantly to 2.0 minutes.

# The Overall Winner: 
# Surprisingly (to me), the absolute fastest way to complete the workload was to skip Ollama entirely and run the programs one at a time (1.3 minutes).


############ Here are the raw timing information
# Not Ollama_2_Subjects: 0.5 minutes, Not Ollama_2_Others_Subjects: 0.8 minutes
# Not Ollama_2_Subjects:  0.9 minutes AND Not Ollama_2_Others_Subjects: 1.1 minutes

# Use Ollama_2_Subjects: 0.8 minutes, Use Ollama_2_Others_Subjects: 0.9 minutes
# Use Ollama_2_Subjects: 0.6 minutes AND Use Ollama_2_Others_Subjects: 0.9 minutes


Llama 3.2-1B MMLU Evaluation (Quantized)

Environment Check
âœ“ Running locally (not in Colab)
âœ“ Platform: Darwin (arm64)
âœ“ Processor: arm
âœ“ Apple Metal (MPS) Available
âœ“ Using Metal Performance Shaders for GPU acceleration
âœ“ Quantization disabled - loading full precision model
âœ“ Hugging Face authenticated

Configuration
Model: meta-llama/Llama-3.2-1B-Instruct
Device: mps
Quantization: None (full precision)
Expected memory: ~2.5 GB (FP16)
Number of subjects: 2


Loading model meta-llama/Llama-3.2-1B-Instruct...
Device: mps
âœ“ Tokenizer loaded
Loading model (this may take 2-3 minutes)...
âœ“ Model loaded successfully!
  Model device: mps:0
  Model dtype: torch.float16
  Running on Apple Metal (MPS)

Starting evaluation on 2 subjects


Progress: 1/2 subjects

Evaluating subject: clinical_knowledge
Testing clinical_knowledge: 100%|█████████████| 265/265 [00:25<00:00, 10.27it/s]
âœ“ Result: 144/265 correct = 54.34%

Progress: 2/2 subjects

Evaluating subject: college_biology


# Task 2

In [6]:
# Followed the steps to use OpenAI LLMs
# Made the OPENAI_API_KEY available like the system env vars.
# Explained the following codes in README.md file of Topic 3.

from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create( model="gpt-4o-mini",
                                          messages=[{"role": "user", "content": "Say: Working!"}],
                                          max_tokens=5)

print('Everything is working fine!')

Everything is working fine!


# Task 3

In [16]:
"""
Manual Tool Calling Exercise (Custom Functions)
Students will see how tool calling works under the hood using explicit routing.
"""

import json
import math
from openai import OpenAI

# ============================================
# PART 1: Define Your Tools
# ============================================

def get_weather(location: str) -> str:
    """Get the current weather for a location"""
    weather_data = {
        "San Francisco": "Sunny, 72°F",
        "New York": "Cloudy, 55°F",
        "London": "Rainy, 48°F",
        "Tokyo": "Clear, 65°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}")

def custom_calculator(operation: str, x: float, y: float = None) -> str:
    """
    Custom calculator handling specific operations.
    No external evaluation libraries used.
    """
    try:
        if operation == "add":
            result = x + y
        elif operation == "subtract":
            result = x - y
        elif operation == "multiply":
            result = x * y
        elif operation == "divide":
            if y == 0:
                return json.dumps({"error": "Cannot divide by zero"})
            result = x / y
        elif operation == "circle_area":
            # x represents the radius
            result = math.pi * (x ** 2)
        elif operation == "sine":
            # x represents the angle in radians
            result = math.sin(x)
        else:
            return json.dumps({"error": f"Unsupported operation: {operation}"})
            
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})

# ============================================
# PART 2: Describe Tools to the LLM
# ============================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g. San Francisco"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "custom_calculator",
            "description": "Evaluate mathematical and geometric operations. Use this tool for ALL math.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide", "circle_area", "sine"],
                        "description": "The specific math operation to perform."
                    },
                    "x": {
                        "type": "number",
                        "description": "The first operand, or the primary parameter (e.g., radius for circle_area, angle in radians for sine)."
                    },
                    "y": {
                        "type": "number",
                        "description": "The second operand. Required for basic arithmetic, but omit for single-variable functions like sine or circle_area."
                    }
                },
                "required": ["operation", "x"]
            }
        }
    }
]

# ============================================
# PART 3: The Agent Loop
# ============================================

def run_agent(user_query: str):
    client = OpenAI()
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant. NEVER calculate math yourself; always route it to the custom_calculator tool."},
        {"role": "user", "content": user_query}
    ]
    
    print(f"User: {user_query}\n")
    
    for iteration in range(5):
        print(f"--- Iteration {iteration + 1} ---")
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto" 
        )
        
        assistant_message = response.choices[0].message
        
        if assistant_message.tool_calls:
            print(f"LLM wants to call {len(assistant_message.tool_calls)} tool(s)")
            messages.append(assistant_message)
            
            for tool_call in assistant_message.tool_calls:
                function_name = tool_call.function.name
                # Parse input arguments via json.loads
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"  Tool: {function_name}")
                print(f"  Args: {function_args}")
                
                # MANUAL DISPATCH
                if function_name == "get_weather":
                    result = get_weather(**function_args)
                elif function_name == "custom_calculator":
                    result = custom_calculator(**function_args)
                else:
                    result = f"Error: Unknown function {function_name}"
                
                print(f"  Result: {result}")
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result
                })
            print()
            
        else:
            print(f"Assistant: {assistant_message.content}\n")
            return assistant_message.content
            
    return "Max iterations reached"

# ============================================
# PART 4: Test It
# ============================================

if __name__ == "__main__":
    print("="*60)
    print("TEST 1: Area of a circle")
    print("="*60)
    run_agent("What is the area of a circle with a radius of 5?")
    
    print("\n" + "="*60)
    print("TEST 2: Sine function")
    print("="*60)
    run_agent("What is the sine of 0.523 radians?")

TEST 1: Area of a circle
User: What is the area of a circle with a radius of 5?

--- Iteration 1 ---
LLM wants to call 1 tool(s)
  Tool: custom_calculator
  Args: {'operation': 'circle_area', 'x': 5}
  Result: {"result": 78.53981633974483}

--- Iteration 2 ---
Assistant: The area of a circle with a radius of 5 is approximately 78.54 square units.


TEST 2: Sine function
User: What is the sine of 0.523 radians?

--- Iteration 1 ---
LLM wants to call 1 tool(s)
  Tool: custom_calculator
  Args: {'operation': 'sine', 'x': 0.523}
  Result: {"result": 0.4994813555186418}

--- Iteration 2 ---
Assistant: The sine of 0.523 radians is approximately 0.4995.



# Task 4

In [21]:
"""
Tool Calling with LangChain
Shows how LangChain abstracts tool calling with parallel and sequential chaining.
"""

import json
import math
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# ============================================
# PART 1: Define Your Tools
# ============================================

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location"""
    weather_data = {
        "San Francisco": "Sunny, 72°F",
        "New York": "Cloudy, 55°F",
        "London": "Rainy, 48°F",
        "Tokyo": "Clear, 65°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}")

@tool
def custom_calculator(operation: str, x: float, y: float = 0.0) -> float:
    """
    Evaluate mathematical operations. Use this tool for ALL math.
    Operations allowed: 'add', 'subtract', 'multiply', 'divide', 'circle_area' (x is radius), 'sine' (x is radians).
    """
    if operation == "add": return x + y
    elif operation == "subtract": return x - y
    elif operation == "multiply": return x * y
    elif operation == "divide": return x / y if y != 0 else 0.0
    elif operation == "circle_area": return math.pi * (x ** 2)
    elif operation == "sine": return math.sin(x)
    return 0.0

@tool
def count_letter(letter: str, text: str) -> int:
    """Counts the number of occurrences of a specific letter in a piece of text. Case-insensitive."""
    return text.lower().count(letter.lower())

@tool
def get_city_elevation(city: str) -> float:
    """Get the elevation of a city in meters."""
    elevations = {
        "San Francisco": 16.0,
        "New York": 10.0,
        "Denver": 1609.0,
        "London": 11.0
    }
    return elevations.get(city, 0.0)

# ============================================
# PART 2: Create LLM with Tools
# ============================================

llm = ChatOpenAI(model="gpt-4o-mini")

# Modern tool mapping dispatch approach
tools = [get_weather, custom_calculator, count_letter, get_city_elevation]
tool_map = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

# ============================================
# PART 3: The Agent Loop
# ============================================

def run_agent(user_query: str):
    """Simple agent showing the manual loop LangGraph automates."""
    
    messages = [
        SystemMessage(content="You are a helpful assistant. Use the provided tools when needed. ALWAYS use the defined tools (for example, custom_calculator for math, count_letter for counting letters). NEVER calculate in your head."),
        HumanMessage(content=user_query)
    ]
    
    print(f"User: {user_query}\n")
    
    # Agent loop - limited to 5 iterations
    for iteration in range(5):
        print(f"--- Iteration {iteration + 1} ---")
        
        response = llm_with_tools.invoke(messages)
        
        if response.tool_calls:
            print(f"LLM wants to call {len(response.tool_calls)} tool(s)")
            messages.append(response)
            
            for tool_call in response.tool_calls:
                function_name = tool_call["name"]
                function_args = tool_call["args"]
                
                print(f"  Tool: {function_name}")
                print(f"  Args: {function_args}")
                
                # Dynamic Tool Dispatch
                if function_name in tool_map:
                    result = tool_map[function_name].invoke(function_args)
                else:
                    result = f"Error: Unknown function {function_name}"
                
                print(f"  Result: {result}")
                
                messages.append(ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                ))
            print()
            
        else:
            print(f"Assistant: {response.content}\n")
            return response.content
            
    print("System Warning: Max iterations reached\n")
    return "Max iterations reached"

# ============================================
# PART 4: Test It
# ============================================

if __name__ == "__main__":
    print("="*60)
    print("TEST 1: Parallel Tool Calls")
    print("="*60)
    run_agent("Are there more i's than s's in Mississippi riverboats?")
    
    print("\n" + "="*60)
    print("TEST 2: Sequential Chaining (Parallel -> Sequential)")
    print("="*60)
    run_agent("What is the sine of the difference between the number of i's and the number of s's in Mississippi riverboats?")
    
    print("\n" + "="*60)
    print("TEST 3: Using All Tools in One Query")
    print("="*60)
    run_agent("Get the weather of New York. Count the number of 'e's in that weather string. Multiply that count by New York's elevation.")

    print("\n" + "="*60)
    print("TEST 4: Hitting the 5 Turn Limit")
    print("="*60)
    run_agent("Start with the number 1. Add 1 to it using the calculator. Take that exact result and add 1 to it using the calculator again. Repeat this exact sequential process 6 times, strictly waiting for each tool result before doing the next tool call.")

TEST 1: Parallel Tool Calls
User: Are there more i's than s's in Mississippi riverboats?

--- Iteration 1 ---
LLM wants to call 2 tool(s)
  Tool: count_letter
  Args: {'letter': 'i', 'text': 'Mississippi riverboats'}
  Result: 5
  Tool: count_letter
  Args: {'letter': 's', 'text': 'Mississippi riverboats'}
  Result: 5

--- Iteration 2 ---
Assistant: In the phrase "Mississippi riverboats," there are 5 occurrences of the letter 'i' and 5 occurrences of the letter 's'. Therefore, there are not more 'i's than 's's; they are equal.


TEST 2: Sequential Chaining (Parallel -> Sequential)
User: What is the sine of the difference between the number of i's and the number of s's in Mississippi riverboats?

--- Iteration 1 ---
LLM wants to call 2 tool(s)
  Tool: count_letter
  Args: {'letter': 'i', 'text': 'Mississippi riverboats'}
  Result: 5
  Tool: count_letter
  Args: {'letter': 's', 'text': 'Mississippi riverboats'}
  Result: 5

--- Iteration 2 ---
LLM wants to call 1 tool(s)
  Tool: custom_c

# Task 5

In [19]:
"""
LangGraph Tool Agent (Task 5)
Demonstrates nodes, edges, memory persistence, and state recovery.
"""

import math
from typing import Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ============================================
# PART 1: Define Tools
# ============================================

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location"""
    weather_data = {
        "San Francisco": "Sunny, 72°F",
        "New York": "Cloudy, 55°F",
        "London": "Rainy, 48°F",
        "Denver": "Snowy, 30°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}")

@tool
def custom_calculator(operation: str, x: float, y: float = 0.0) -> float:
    """
    Evaluate mathematical operations. Use this tool for ALL math.
    Operations allowed: 'add', 'subtract', 'multiply', 'divide', 'circle_area' (x is radius), 'sine' (x is radians).
    """
    if operation == "add": return x + y
    elif operation == "subtract": return x - y
    elif operation == "multiply": return x * y
    elif operation == "divide": return x / y if y != 0 else 0.0
    elif operation == "circle_area": return math.pi * (x ** 2)
    elif operation == "sine": return math.sin(x)
    return 0.0

@tool
def count_letter(letter: str, text: str) -> int:
    """Counts the number of occurrences of a specific letter in a piece of text. Case-insensitive."""
    return text.lower().count(letter.lower())

@tool
def get_city_elevation(city: str) -> float:
    """Get the elevation of a city in meters."""
    elevations = {"San Francisco": 16.0, "New York": 10.0, "Denver": 1609.0, "London": 11.0}
    return elevations.get(city, 0.0)

tools = [get_weather, custom_calculator, count_letter, get_city_elevation]

# ============================================
# PART 2: Define State & Graph Nodes
# ============================================

# The State uses add_messages to append new messages rather than overwriting
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def call_model(state: AgentState):
    """Node that invokes the LLM"""
    messages = state["messages"]
    
    # Ensure system prompt is present at the start of the context
    if not any(isinstance(m, SystemMessage) for m in messages):
        sys_msg = SystemMessage(
            content="You are a helpful assistant. Use the provided tools. ALWAYS use custom_calculator for math."
        )
        messages = [sys_msg] + messages
        
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# LangGraph's prebuilt ToolNode handles the tool mapping and dispatch automatically
tool_node = ToolNode(tools)

# ============================================
# PART 3: Build Graph with Checkpointing
# ============================================

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Add edges
workflow.add_edge(START, "agent")
# The conditional edge routes to "tools" if the LLM made a tool call, otherwise it routes to END
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

# Initialize memory checkpointer for state persistence
memory = MemorySaver()

# Compile the graph
app = workflow.compile(checkpointer=memory)

# ============================================
# PART 4: Execution & Testing Function
# ============================================

def chat_with_agent(user_query: str, config: dict):
    print(f"\nUser: {user_query}")
    print("-" * 50)
    
    # We use stream to observe the internal reasoning and tool usage steps
    events = app.stream(
        {"messages": [HumanMessage(content=user_query)]},
        config=config,
        stream_mode="values" 
    )
    
    for event in events:
        last_msg = event["messages"][-1]
        
        # Filter stream to only print new activity
        if isinstance(last_msg, AIMessage):
            if last_msg.tool_calls:
                print(f"Agent Action: Calling {len(last_msg.tool_calls)} tool(s)...")
                for tc in last_msg.tool_calls:
                    print(f"  -> {tc['name']}({tc['args']})")
            elif last_msg.content:
                print(f"Agent: {last_msg.content}")
        elif isinstance(last_msg, ToolMessage):
            print(f"Tool Result ({last_msg.name}): {last_msg.content}")

if __name__ == "__main__":
    # "A thread is a unique ID or thread identifier assigned to each checkpoint saved by a checkpointer."
    thread_config = {"configurable": {"thread_id": "portfolio_conversation"}}
    
    print("============================================================")
    print("TEST 1: Tool Use (Starting the conversation)")
    print("============================================================")
    chat_with_agent("What's the weather like in Denver?", thread_config)
    
    print("\n============================================================")
    print("TEST 2: Conversation Context (Memory)")
    print("============================================================")
    # The agent should remember we are talking about Denver's weather
    chat_with_agent("How many 'w's are in that weather report string?", thread_config)
    
    print("\n============================================================")
    print("TEST 3: State Recovery / Time Travel")
    print("============================================================")
    # Fetch all historical checkpoints for this thread
    history = list(app.get_state_history(thread_config))
    
    # We will iterate back to find the checkpoint right before the user asked the 2nd question
    # We will iterate back to find the checkpoint right before the user asked the 2nd question
    target_config = None
    for state_snapshot in history:
        # Safely fetch the messages list, defaulting to an empty list if missing
        messages = state_snapshot.values.get("messages", [])
        
        # Only check the last message if the list is not empty
        if messages:
            last_message = messages[-1]
            # Look for the exact AI response from Test 1
            if isinstance(last_message, AIMessage) and "Snowy" in last_message.content:
                target_config = state_snapshot.config
                break
            
    if target_config:
        print("\n[System: Recovering previous state... Ignoring the letter 'w' question.]")
        print("[System: Branching conversation from the first answer:]")
        chat_with_agent("Actually, grab Denver's elevation and calculate the sine of it instead.", target_config)

TEST 1: Tool Use (Starting the conversation)

User: What's the weather like in Denver?
--------------------------------------------------
Agent Action: Calling 1 tool(s)...
  -> get_weather({'location': 'Denver'})
Tool Result (get_weather): Snowy, 30°F
Agent: The weather in Denver is snowy with a temperature of 30°F.

TEST 2: Conversation Context (Memory)

User: How many 'w's are in that weather report string?
--------------------------------------------------
Agent Action: Calling 1 tool(s)...
  -> count_letter({'letter': 'w', 'text': 'The weather in Denver is snowy with a temperature of 30°F.'})
Tool Result (count_letter): 3
Agent: There are 3 occurrences of the letter 'w' in the weather report string.

TEST 3: State Recovery / Time Travel


# Task 6

In [ ]:
# I think there was an opportunity for parallelization in using the tools. 
# For instance, if there were tasks where several tools are needed to execute, I think, parallelization would be great there.

# Acknowledgement

In [ ]:
# I acknowledge that I have used GenAI very significantly to do the assignments/tasks of the topics. 
# However, I tried to understand the codes. Also, it is not that I just copied all tasks of the topic and pasted the outputs here. 
# I went step by step of the topic by understanding and debugging the things. 
# Moreover, I spent several hours to understand the things and complete the tasks per topic.
# Besides, I checked whether the results make sense.

# Here are the prompts I used for this topic:

# https://chatgpt.com/share/69a54900-6d98-8012-ac25-f354de093940
# https://gemini.google.com/share/3519641be2c2